In [ ]:
%xmode Context
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt

import sys
from pathlib import Path

# help locating the package sgpykit (comment this out if you installed it)
PKG_PARENT = Path().resolve().parent
sys.path.insert(0, str(PKG_PARENT))

import sgpykit as sg

import logging
from sgpykit.util.log import logger

logging.basicConfig(
    # see https://docs.python.org/3/library/logging.html#logrecord-attributes
    format="%(levelname)s %(name)s:%(lineno)d: %(message)s",
)
# switching log level for sgpykit. Also see https://docs.python.org/3/library/logging.html#logging-levels
logger.setLevel(logging.INFO)
# logger.setLevel(logging.DEBUG)

# PART 4: Interpolation on a Sparse Grid

## Basics

The sparse grid also provides an interpolant / surrogate model for the original function. The
interpolant can be evaluated at non-grid points.
All the previous topics (changing the domain, building anisotropic grids, etc.) apply immediately to
the interpolation case.

In [ ]:
#f = lambda x,b: np.prod(1.0 / np.sqrt(x + b), axis=0, keepdims=True)
f = lambda x, b: np.prod((x + b)**-0.5, axis=0)
b = 3
N = 4
w = 8

In [ ]:
# generate the knots and the SM grid. 'nonprob' means we are integrating w.r.t. the pdf rho(x)=1 and not rho(x)=1/prod(b_i - a_i)
knots = lambda n: sg.knots_uniform(n,-1,1,'nonprob')
S,_ = sg.create_sparse_grid(N,w,knots,sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)

#non_grid_points=rand(N,100) # TODO ?
non_grid_points = np.hstack((0.5 * np.ones((N, 1)), np.zeros((N, 1))))

function_on_grid = f(Sr.knots, b)

f_values = sg.interpolate_on_sparse_grid(S, Sr, function_on_grid, non_grid_points)
# compare with exact value
print('Interpolation error:',
      np.max( np.abs( f_values-f(non_grid_points,b) ) ))

## Interpolation on a Tensor Grid

In [ ]:
#f = lambda x,b: np.prod(1.0 / np.sqrt(x + b), axis=0, keepdims=True)
f = lambda x, b: np.prod((x + b)**-0.5, axis=0)
b = 3
N = 2
# let's build a tensor grid with the following choices
idx = [10, 8]
knots = lambda n: sg.knots_uniform(n,-1,1,'nonprob')
lev2knots = sg.lev2knots_lin

In [ ]:
# create the tensor grid, convert it to sparse
T = sg.tensor_grid(N, lev2knots(idx), knots)
S = sg.tensor_to_sparse(T)
Sr = sg.reduce_sparse_grid(S)

In [ ]:
non_grid_points = np.hstack((0.5 * np.ones((N, 1)), np.zeros((N, 1))))

function_on_grid = f(Sr.knots, b)

f_values = sg.interpolate_on_sparse_grid(S, Sr, function_on_grid, non_grid_points)

# compare with exact value
print('Interpolation error:',
      np.max( np.abs( f_values-f(non_grid_points,b) ) ))  # matlab result: 6.411494807290197e-08

## Interpolation in 1D

For 1D interpolation, a dedicated function exists. It's called [`univariate_interpolant()`](https://uncertaintyhub.github.io/sgpykit-doc/_autosummary/sgpykit.tools.polynomials_functions.html#sgpykit.tools.polynomials_functions.univariate_interpolant) and can operate on
vector-valued functions, like here below, where we interpolate a function with two components, i.e., $F: \mathbb{R} \to \mathbb{R}^2$.

In [ ]:
# the two components of f
f1 = lambda x: x**3
f2 = lambda x: np.sin(2*x)

# interpolation points and values
x_interp = np.linspace(-1, 2, 4)
F_interp = np.vstack([f1(x_interp),
                      f2(x_interp)])

# evaluate the interpolant on a much finer grid
x_eval = np.arange(-1, 2.01, 0.01)
F_eval_interp = sg.univariate_interpolant(x_interp,F_interp,x_eval)

In [ ]:
fig, axs = sg.figure_create(nrows=1, ncols=2, figsize=(8, 4))
sg.plot(axs[0], x_eval, F_eval_interp[0,:], 'DisplayName', 'interpolant')
sg.plot(axs[0], x_interp, F_interp[0,:],'o','DisplayName','interpolation points')
sg.plot(axs[0], x_eval, f1(x_eval),'DisplayName','true fun')
axs[0].legend()
sg.plot(axs[1], x_eval, F_eval_interp[1,:], 'DisplayName', 'interpolant')
sg.plot(axs[1], x_interp, F_interp[1,:],'o','DisplayName','interpolation points')
sg.plot(axs[1], x_eval, f2(x_eval),'DisplayName','true fun')
axs[1].legend()

fig.tight_layout() 

## Interpolation Error on Sparse Grid Points

Since the sparse grid is a linear combination of several tensor grid interpolants, the interpolation
error at a point of the sparse grid is not necessarily zero, unless all tensor interpolants
include that point.

In [ ]:
#f = lambda x,b: np.prod(1.0 / np.sqrt(x + b), axis=0, keepdims=True)
f = lambda x, b: np.prod((x + b)**-0.5, axis=0)
b = 3
N = 4
# a sparse grid withx non-nested points: interpolation error in sparse grid points will be
# non-zero in general
w = 4

knots = lambda n: sg.knots_uniform(n,-1,1,'nonprob')
S,_ = sg.create_sparse_grid(N,w,knots,sg.lev2knots_lin)
Sr = sg.reduce_sparse_grid(S)

non_grid_points=np.zeros((N,1))

function_on_grid,*_=sg.evaluate_on_sparse_grid(lambda x: f(x,b), S=None, Sr=Sr)

f_values = sg.interpolate_on_sparse_grid(S,Sr,function_on_grid,non_grid_points)

print('Interpolation error - non-nested grid:')
np.max( np.abs( f_values-f(non_grid_points,b) ) )  # matlab result: 8.556139706267230e-06

In [ ]:
# the interpolation error will instead be zero if we use nested points and
# consider e.g. [0 0 0 0] which belongs to all of the tensor grids

knots = lambda n: sg.knots_CC(n,-1,1,'nonprob')
T,_ = sg.create_sparse_grid(N,w,knots,sg.lev2knots_doubling)
Tr = sg.reduce_sparse_grid(T)

non_grid_points = np.zeros((N,1))
function_on_grid = f(Tr.knots,b)

f_values = sg.interpolate_on_sparse_grid(T,Tr,function_on_grid,non_grid_points)

print('Interpolation error - nested grid:')
np.max( np.abs( f_values-f(non_grid_points,b) ) )

## Plot Sparse Grids Interpolant
### Case $N=2$

In [ ]:
# define sparse grid over [4,6] x [1,5]
N=2
aa=[4, 1]
bb=[6, 5]

# the function to be interpolated
f = lambda x: 1./(1+0.5*sum(x**2))
# create a sparse grid and evaluate the function on it
domain = np.vstack((aa, bb))
knots1 = lambda n: sg.knots_CC(n,aa[0],bb[0],'nonprob')
knots2 = lambda n: sg.knots_CC(n,aa[1],bb[1],'nonprob')
w = 4
S,_ = sg.create_sparse_grid(N,w,[knots1,knots2],sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)

values_on_grid,*_=sg.evaluate_on_sparse_grid(f, S=None, Sr=Sr)

In [ ]:
#the plot: several examples of usage
fig, axs = sg.figure_create(dims=3)
sg.plot_sparse_grids_interpolant(axs, S, Sr, domain, values_on_grid, 'with_f_values')
axs.view_init(elev=16, azim=270+200)  # add 270 to have view like in matlab

In [ ]:
fig, axs = sg.figure_create(dims=3)
sg.plot_sparse_grids_interpolant(axs, S, Sr, domain, values_on_grid, nb_plot_pts=10)
axs.view_init(elev=16, azim=270+200)  # add 270 to have view like in matlab

In [ ]:
# access to plot handles for further editing is available. E.g., this sets dots to black

fig, axs = sg.figure_create(dims=3)
h = sg.plot_sparse_grids_interpolant(axs, S, Sr, domain, values_on_grid, 'with_f_values')
axs.view_init(elev=16, azim=270+200)  # add 270 to have view like in matlab
# Check if h is a list of line objects or a single line object
if isinstance(h, list):
    for line in h:
        line.set_markerfacecolor('black')
else:
    h.set_markerfacecolor('black')

In [ ]:
fig, axs = sg.figure_create()
sg.plot_sparse_grid(axs, Sr, [])

In [ ]:
## as expected, the interpolant might be bad if equispaced point are used

f = lambda x: 1./(1+(5*x[0])**2)*1./(1+(5*x[1])**2)
a=-1; b=1;
domain = np.array([[-1, -1], [1, 1]])
N = 2
w = 6
lev2knots = sg.lev2knots_doubling

# a bad choice: equispaced points, we get them by trap-rule. Build a grid with them
knots_bad = lambda n: sg.knots_trap(n,a,b,'nonprob')
S_bad,_ = sg.create_sparse_grid(N,w,knots_bad,lev2knots)
Sr_bad = sg.reduce_sparse_grid(S_bad)
f_values_bad,*_ = sg.evaluate_on_sparse_grid(f,S=None, Sr=Sr_bad)

# a good choice: CC points. Build another grid with them, to compare. Note that the two grids will have the same
# number of points
knots_ok = lambda n: sg.knots_CC(n,a,b,'nonprob')
S_ok,_ = sg.create_sparse_grid(N,w,knots_ok,lev2knots)
Sr_ok = sg.reduce_sparse_grid(S_ok)
f_values_ok,*_ = sg.evaluate_on_sparse_grid(f,S=None, Sr=Sr_ok)

In [ ]:
# a subplot figure where we compare the two grids and the two interpolants
fig, axs = sg.figure_create()
sg.plot_sparse_grid(axs, Sr_bad,[],'o','MarkerSize',4,'LineWidth',2)

In [ ]:
# a subplot figure where we compare the two grids and the two interpolants
fig, axs = sg.figure_create(dims=3)
sg.plot_sparse_grids_interpolant(axs, S_bad,Sr_bad,domain,f_values_bad,'with_f_values',nb_plot_pts=40)
axs.view_init(elev=29, azim=240)

In [ ]:
# a subplot figure where we compare the two grids and the two interpolants
fig, axs = sg.figure_create()
sg.plot_sparse_grid(axs, Sr_ok,[],'o','MarkerSize',4,'LineWidth',2)

In [ ]:
# a subplot figure where we compare the two grids and the two interpolants
fig, axs = sg.figure_create(dims=3)
sg.plot_sparse_grids_interpolant(axs, S_ok,Sr_ok,domain,f_values_ok,'with_f_values',nb_plot_pts=40)
axs.view_init(elev=29, azim=240)

### Case $N=3$

In [ ]:
# define sparse grid over [4,6] x [1,5] x [2 3]
N=3
aa=[4, 1, 2]
bb=[6, 5, 3]

# the function to be interpolated
f = lambda x: 1./(1+0.5*sum(x**2))

# create a sparse grid and evaluate the function on it
domain = np.vstack((aa, bb))
knots1 = lambda n: sg.knots_CC(n,aa[0],bb[0],'nonprob')
knots2 = lambda n: sg.knots_CC(n,aa[1],bb[1],'nonprob')
knots3 = lambda n: sg.knots_CC(n,aa[2],bb[2],'nonprob')
w = 4
S,_ = sg.create_sparse_grid(N,w,[knots1,knots2,knots3],sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)

values_on_grid,*_=sg.evaluate_on_sparse_grid(f, S=None, Sr=Sr)

In [ ]:
#the plot: several examples of usage
fig, axs = sg.figure_create(dims=3)
sg.plot_sparse_grids_interpolant(axs, S, Sr, domain, values_on_grid)

In [ ]:
#the plot: several examples of usage
fig, axs = sg.figure_create(dims=3)
sg.plot_sparse_grids_interpolant(axs, S, Sr, domain, values_on_grid, 'with_f_values','nb_plot_pts',10,'nb_contourfs',10,'nb_contourf_lines',40)
# axs.view_init(elev=16, azim=270+200)  # add 270 to have view like in matlab

In [ ]:
# we have two ways of plotting the sparse grid:
sg.plot3_sparse_grid(Sr, [], 'color', 'k', 'marker', 'o', 'MarkerFaceColor', 'k')

In [ ]:
# way 2): two-dimensional projections (they will look identical in this case)
fig, axs = sg.figure_create(nrows=1, ncols=3, figsize=(8, 4))
sg.plot_sparse_grid(axs[0], Sr, [1,2])
sg.plot_sparse_grid(axs[1], Sr, [2,3])
sg.plot_sparse_grid(axs[2], Sr, [1,3])
fig.tight_layout()  # auto spacing subplots

### Case $N>3$

In [ ]:
N = 7
aa = -1 * np.ones(N)
bb = np.ones(N)

# The function to be interpolated
#f = lambda x:
def f(x):
    x = np.atleast_2d(x)
    y = 1 / (1 + 0.5 * x[0, :]**2 + 0.25 * x[1, :]**2 + 5 * x[2, :]**2 + 
                  2 * x[3, :]**2 + 0.001 * x[4, :]**2 + 10 * x[5, :]**2 + 10 * x[6, :]**2)
    return y
# f = lambda x: 1 / (1 + 0.5 * x[0, :]**2 + 0.5 * x[1, :]**2 + 0.5 * x[2, :]**2 + 
#                  0.5 * x[3, :]**2 + 0.5 * x[4, :]**2 + 0.5 * x[5, :]**2 + 0.5 * x[6, :]**2)

domain = np.vstack((aa, bb))
knots = lambda n: sg.knots_CC(n,-1,1,'nonprob')
w=6
S,_ = sg.create_sparse_grid(N, w, knots, sg.lev2knots_doubling)
Sr = sg.reduce_sparse_grid(S)

values_on_grid,*_=sg.evaluate_on_sparse_grid(f, S=None, Sr=Sr)

In [ ]:
# add f_values. Note that there are possibly several points which share the values of the coordinates in the cuts,
# therefore there will be points not on the surface. This helps understanding the fluctuations of the function
# when the coordinates not in the cut are not fixed to their average value. In this specific example, changing the
# values of the frozen variables from their averages happens to lower the value of the function. The function generates
# one new figure per cut

fig, axs = sg.figure_create(nrows=1, ncols=3, figsize=(16, 12), dims=3)
sg.plot_sparse_grids_interpolant(axs, S, Sr, domain, values_on_grid, 'with_f_values')

In [ ]:
# specify cuts. Again, because we are specifying cuts, a new figure per cut is generated.  The code below generates two figures
# (the first one is empty)
fig, axs = sg.figure_create(nrows=1, ncols=2, figsize=(16, 12), dims=3)
sg.plot_sparse_grids_interpolant(axs, S, Sr, domain, values_on_grid,'two_dim_cuts',[1, 4, 2, 7])

In [ ]:
# we have two ways of plotting the sparse grid:
sg.plot3_sparse_grid(Sr, [1,2,4], 'color', 'k', 'marker', 'o', 'MarkerFaceColor', 'k')

In [ ]:
# way 2): two-dimensional projections
fig, axs = sg.figure_create()
sg.plot_sparse_grid(axs, Sr, [1,2])

## Convergence Study

See test_sparse_interpolation.m. (not ported yet)

## NEXT: Miscellaneous

- Compute the g-PCE of a Function Given its Sparse Grid Approximation
- Sparse-Grids-Based Sensitivity Analysis
- Save a Sparse Grid to a File

=> **[Go to tutorial "Misc"](04-misc.ipynb)**